# Tokopedia Scraper Ultimate (GQL Mobile Payload)
Notebook ini 100% menggunakan *request* murni (cURL-like) dari hasil ekstrak DevTools Anda. Ini adalah cara yang sepenuhnya kebal terhadap Akamai Bot Manager dan Schema Validation Tokopedia.

In [7]:
%pip install curl-cffi pandas tqdm -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import random
import pandas as pd
import re
from curl_cffi import requests
from tqdm.notebook import tqdm

# ==========================================
# KONFIGURASI PENCARIAN
# ==========================================
KEYWORDS = [
    "laptop",
    "keyboard wireless",
    "headset gaming",
    "handphone samsung",
    "skincare wajah",
    "sepatu sneakers",
    "tas ransel",
    "kamera mirrorless",
    "charger powerbank",
    "baju kaos polos",
    "mouse komputer",
    "earphone bluetooth",
    "jam tangan pria",
    "parfum wanita",
    "rice cooker"
]

TARGET_TOTAL_ROWS = 50000
TARGET_PER_KEYWORD = TARGET_TOTAL_ROWS // len(KEYWORDS)  # ~1800 baris per keyword

# MENGGUNAKAN SORT OPTIONS UNTUK BYPASS PAGINATION (Halaman 2)
# Tokopedia API V5 mensyaratkan token cursor yang sangat rumit untuk pindah halaman,
# jadi kita mengakali dengan cara mengganti SORT (Urutkan) pada Halaman 1 agar mendapatkan produk berbeda!
SORT_OPTIONS = [
    23, # Paling Sesuai
    9,  # Terlaris
    3,  # Ulasan Terbanyak
    4,  # Terbaru
    8,  # Harga Tertinggi
    5   # Harga Terendah
]

REVIEWS_PER_PRODUCT = 40 # Ambil hingga 40 ulasan per produk
OUTPUT_FILE = "dataset\raw_tokopedia_dataset.csv"

print("Library dimuat. Target Total Baris:", TARGET_TOTAL_ROWS)
print("Target Baris Per Kategori:", TARGET_PER_KEYWORD)

Library dimuat. Target Total Baris: 50000
Target Baris Per Kategori: 3333


## Konfigurasi Kredensial
Header dan Cookie diambil langsung dari *payload* milik Anda.

In [9]:
HEADERS = {
    'sec-ch-ua-platform': '"Windows"',
    'sec-ch-ua': '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
    'x-price-center': 'true',
    'bd-device-id': '7647226487314482689',
    'sec-ch-ua-mobile': '?0',
    'accept': '*/*',
    'content-type': 'application/json',
    'bd-web-id': '7647226487314482689',
    'x-version': '7149a47',
    'x-source': 'tokopedia-lite',
    'x-dark-mode': 'false',
    'tkpd-userid': '12787827',
    'x-device': 'mobile',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36',
    'x-tkpd-lite-service': 'phoenix',
    'cookie': 'DID_JS=MDEyMWU1NTllMGU2ZDNlZjY3MTJmMTdlMmQ3OGY1ODk1N2Y2MWUwZTdiM2M4ZmRlMDQ1YmI1NjM3ZWMwN2VlODVmN2I5NjNjNDU1NTUxZmNkOWMwZWNhMDJmOTFhMTcx47DEQpj8HBSa+/TImW+5JCeuQeRkm5NMpJWZG3hSuFU=; serverECT=4g; _UUID_NONLOGIN_=97ec507479730278bd5ee2a7d3c0305d; _UUID_NONLOGIN_.sig=GwR6-vJPesYKhg1ISqjlWQmZcgc; _gcl_au=1.1.262415552.1780508667; _ga=GA1.1.1121156258.1780508667; __krpc=true; bd-device-id=7647226487314482689; hfv_banner=true; ect=4g; home_libra=%255B%257B%2522experiment%2522%253A%2522home_cache_exp%2522%252C%2522variant%2522%253A%2522variant1%2522%257D%252C%257B%2522experiment%2522%253A%2522low_price_nonlogin%2522%252C%2522variant%2522%253A%2522var1%2522%257D%252C%257B%2522experiment%2522%253A%2522home_revamp_3_type%2522%252C%2522variant%2522%253A%2522variant1%2522%257D%255D; w2a-hide=1; g_state={"i_l":0,"i_ll":1780748702797,"i_e":{"enable_itp_optimization":0},"i_et":1780508668358}; gec_id=54923298751921856; uidh=FWryhvesGChoNnqBubkcVS5FBtLDii3/6cVMtr6xaTU=; tkpd-x-device=undefined; FPF=1; tuid=12787827; uide=hOOaNeoWaNxLKO04g2VFPE5PuhTAC0AFpu8OfV2ff+ne/snN; dt_intl=DVLT_6542b775e3263c27e321b929-f52fc6e0_dFlt; aus=1; _CASE_=277e3d15387e667e7e707e3f15387e666d6b6a707e3815387e666e6e6b68707e30092c387e667e7e707e303d287e667e6c7e707e303e307e667e163d373d2e283d7c0c292f3d287e707e3033323b7e667e6c7e707e2c1f337e667e7e707e2f15387e667e6c7e707e2f08252c397e667e7e707e2b15387e667e6c7e707e2b342f2f7e667e07017e21; _UUID_CAS_=caddce49-52cb-4d49-9f4d-005cf253da00; show_adult=false; srp_desktop_show_sensitive_images=false; bm_sz=792D71E0426918AB7378B4662C81BBEF~YAAQ1QkgF170rW+eAQAA25PtnABht+SsD3KRWFc3zJhYLcZ/n0xoh8pp/atjoQLY6CWdvPu0lUjEsmSrc8Ol+u0cOdvZ0oxtXKH6zD09ledO0wAImX7mRicg+94aslNZu7h9d2XmRhilrODWXLXh9iXZFGpoVjTR2RIepDjXIV45J/EddwKGtnOaXB4ErRUw2hYJa95JJs1lbTZv57A3Cv6O0BPp4a36rPoE1NzX4hfVVM8Fesd9QVpmp/EbvK5ToqYD4wsLDx2xCK8U2LkioNChUNFuqrmdhdjiIeYZHlMSVCk9+SzOP0chgvYhBbAi3bp2+sE2G5GsS2f01CpsjgobFMXX7xVCRCpkKzHEi4OhSUI/EdIR6rsKUHjFFqtjFXu11LaIpB85Wd+81tlcXYTCi78zgONA0xQEFruouy5FUC1Vc4C4stcv5MHcl6d9gqrm2nus8olyp8dr3Q8oNAI=~4403764~4277047; _abck=32066F56C76B50DCDD5B552788A4B547~0~YAAQ1QkgF5z1rW+eAQAA0pntnBCozU9xH+DJ2mAV3g3N1jE2f/DpickgbUZVQBVq5dpyum9qshM6SIeviIRhUDurZ+8IxZ4DdRsxbSC8pI9ZRVzO1lqnncwLHPoZEd7SzX7LcagCB28P0Db7mASZPLs/+HSElJnD6GgvRwKhNf3gqWAYzGtM5pltsPkj5SadveNtQR8MGeN9beT0rL+k2CYCcBFUwKJ0VmuvosARkBfnFb2tHOLhIkJzlLWchA6ZPe3eVAzyJkErid2bRGkr3e5R1v2Z0UESRnyNJp217yr7u6ki3Xt6bzc3xBLrlNa1/efWJrYlq0PWa/bcSJngbc6n7E49lP+su0QNUnu+4kPix4qIc/PuIfDpMdJ+cIJggMFGL3qa5nPPMoa8a8PufqLDht5FG/P/G2O3FEwtZhqtZ/VgNuzssHQlo08Bxem5hfQKb1grqp8E82Ip6sCQnzMjgXTcPtAhB4B1DkXXhWlUXlDrk65CTtnWRdT4UHjXYEDj4hbj6TB8RCIIEKmNEy6JGhSTeOCpycdX+r+jna4311A5rSBSElehHnaruzs0e8kmbEGNLGbKvAHkwJqfBLtTLmLgJjJdf78ih6g9xM06PcAmKbly3KCDBccaKAO6M5ZFOHrtxekcISrzH1TLwHE+/jsIObtF5vgGqnz5QWsr/YEZ0uhmUzx3c3mMXeOhCX/m4CyWlJC6dBJI7MXNjib7yh2iu3VkxKJ0jDRZTRuVwkFu8QwhfBgTdMJRFNgLOyG000aZoPGFR1MRVcjJwVmnZmylN/vXgqvjnLS7BiiTAuiRRKWrLe3o+nI0jhSaVBIb8czFQZhOtDEpzSY2NOH80rczXUypnMZAF3hrO6VT1kftTeVNxtojGM3lklhTsdg2YeN8N6CxRtOUyAtqhPYhpwtIS7kN3n4eMq4SvlYQn1Ir18UvurvZDQkgs5rbNZA+zM04EWN1hI4PSPc.5HVhAoyzBqGnF5ICC+ofxFl0i1XpNuxh26Y7c22333gW08U=~-1~-1~-1~AAQAAAAF%2f%2f%2f%2f%2f0B3ayCzxxbfyzYqc1VV5UUGJ4vyeK8uqGSk8VPuowOp27UPefJu6bjICI5aInOtiUiwIGl1182KHgL0e%2f2l2ExLXaZ6NNhD%2f61RmisIJbu9Vzx6R4qzc+4PaahuH3g3L2ryOf0%3d~-1; srp_show_sensitive_images=false; _ga_70947XW48P=GS2.1.s1780748633$o10$g1$t1780749306$j29$l0$h0'
}

session = requests.Session(impersonate="chrome120")

Q_SEARCH = "query SearchProductV5Query($searchProductV5Param: String!) { searchProductV5(params: $searchProductV5Param) { header { totalData responseCode keywordProcess keywordIntention componentID isQuerySafe additionalParams backendFilters backendFiltersToggle meta { dynamicFields __typename } __typename } data { totalDataText banner { position text url imageURL componentID trackingOption __typename } redirection { url applink __typename } related { relatedKeyword position trackingOption otherRelated { keyword url applink componentID products { oldId: id id: id_str_auto_ name url applink mediaURL { image __typename } shop { oldId: id id: id_str_auto_ name city tier __typename } badge { id title url __typename } price { text number __typename } freeShipping { url __typename } labelGroups { id position title type url styles { key value __typename } __typename } rating wishlist ads { id productClickURL productViewURL productWishlistURL tag __typename } meta { oldWarehouseID: warehouseID warehouseID: warehouseID_str_auto_ componentID oldParentID: parentID parentID: parentID_str_auto_ __typename } __typename } __typename } __typename } suggestion { currentKeyword suggestion query text componentID trackingOption __typename } shopWidget { headline { badge { id title url __typename } shop { id ttsSellerID location City name ratingScore imageShop { sURL __typename } products { id id_str_auto_ ttsProductID name url rating mediaURL { image image300 videoCustom __typename } shop { oldId: id id: id_str_auto_ ttsSellerID name city __typename } price { text number range discountPercentage original __typename } labelGroups { id position title type url styles { key value __typename } __typename } meta { oldParentID: parentID parentID: parentID_str_auto_ isPortrait oldWarehouseID: warehouseID warehouseID: warehouseID_str_auto_ __typename } stock { ttsSKUID __typename } __typename } __typename } __typename } meta { redirect __typename } __typename } ticker { id text query applink componentID trackingOption __typename } violation { headerText descriptionText imageURL ctaURL ctaApplink buttonText buttonType __typename } products { oldId: id id: id_str_auto_ ttsProductID name url applink mediaURL { image image300 videoCustom __typename } shop { oldId: id id: id_str_auto_ ttsSellerID name url city tier __typename } stock { ttsSKUID __typename } badge { id title url __typename } price { text number range original discountPercentage __typename } freeShipping { url __typename } labelGroups { id position title type url styles { key value __typename } __typename } labelGroupsVariant { title type typeVariant hexColor __typename } category { oldId: id id: id_str_auto_ name breadcrumb gaKey __typename } rating wishlist ads { id productClickURL productViewURL productWishlistURL tag __typename } meta { oldParentID: parentID parentID: parentID_str_auto_ oldWarehouseID: warehouseID warehouseID: warehouseID_str_auto_ isImageBlurred isPortrait __typename } __typename } __typename } __typename } }"
Q_REVIEW = "query productReviewList($productURL: String!, $page: Int!, $limit: Int!, $sortBy: String, $filterBy: String, $opt: String) { productrevGetProductReviewList( productID: \"\" productURL: $productURL page: $page limit: $limit sortBy: $sortBy filterBy: $filterBy opt: $opt ) { productID list { id: feedbackID variantName message productRating reviewCreateTime reviewCreateTimestamp isReportable isAnonymous videoAttachments { attachmentID videoUrl __typename } imageAttachments { attachmentID imageThumbnailUrl imageUrl __typename } reviewResponse { message createTime __typename } user { userID fullName image url label __typename } likeDislike { totalLike likeStatus __typename } stats { key formatted count __typename } badRatingReasonFmt __typename } shop { shopID name url image __typename } variantFilter { isUnavailable ticker __typename } hasNext __typename } }"

## Fungsi Ekstraksi GQL
Fungsi ini memanggil API Tokopedia langsung dan mengurai JSON-nya.

In [10]:
def fetch_search(keyword, sort_option=23):
    # MENGGUNAKAN SORT OPTIONS (ob) MENGGANTIKAN PAGE (page)
    param_str = f"device=mobile&enable_lite_deduplication=true&enter_method=normal_search&l_name=sre&navsource=&ob={sort_option}&page=1&q={keyword.replace(' ', '+')}&rows=60&source=search&srp_component_id=02.01.00.00&srp_page_id=&srp_page_title=&unique_id=93ca95a7632139e8de495d9aad46a929&use_page=true&user_addressId=&user_cityId=176&user_districtId=2274&user_id=12787827&user_lat=0&user_long=0&user_postCode=&user_warehouseId=0&warehouses="
    payload = [{
        "operationName": "SearchProductV5Query",
        "variables": {"cursor":"", "searchProductV5Param": param_str},
        "query": Q_SEARCH
    }]
    
    try:
        r = session.post("https://gql.tokopedia.com/graphql/SearchProductV5Query", headers=HEADERS, json=payload, timeout=20)
        data = r.json()
        if isinstance(data, list): data = data[0]
        if "errors" not in data:
            return data.get("data",{}).get("searchProductV5",{}).get("data",{}).get("products",[])
        else:
            print("Search API Error:", data["errors"][0]["message"])
    except Exception as e:
        print("HTTP Error:", e)
    return []

def fetch_reviews(product_url, limit=40):
    # WAJIB membersihkan URL dari tracking parameter (?extParam=...) agar API Review mau memprosesnya!
    clean_url = product_url.split("?")[0]
    
    payload = [{
        "operationName": "productReviewList",
        "variables": {
            "productURL": clean_url,
            "page": 1, "limit": limit, "sortBy": "informative_score desc", "filterBy": "", "opt": ""
        },
        "query": Q_REVIEW
    }]
    try:
        r = session.post("https://gql.tokopedia.com/graphql/productReviewList", headers=HEADERS, json=payload, timeout=20)
        data = r.json()
        if isinstance(data, list): data = data[0]
        if "errors" not in data:
            return data.get("data",{}).get("productrevGetProductReviewList",{})
    except Exception:
        pass
    return {}

In [11]:
all_records = []
seen_products = set() # Untuk mencegah duplikat saat kita mengganti sort options

for keyword in KEYWORDS:
    print(f"\n=== Ekstraksi Keyword: {keyword} ===")
    
    # Counter jumlah ulasan spesifik untuk keyword ini
    keyword_reviews_count = 0
    
    for sort_opt in SORT_OPTIONS:
        if keyword_reviews_count >= TARGET_PER_KEYWORD:
            print(f"[✓] Target {TARGET_PER_KEYWORD} baris untuk '{keyword}' sudah tercapai!")
            break
            
        products = fetch_search(keyword, sort_option=sort_opt)
        print(f"Mode Sort {sort_opt}: Ditemukan {len(products)} produk.")
        
        for idx, prod in enumerate(products):
            if keyword_reviews_count >= TARGET_PER_KEYWORD:
                break
                
            p_name = prod.get("name", "")
            p_url = prod.get("url", "")
            
            # Cegah duplikasi jika produk ini sudah pernah kita scrape di mode Sort sebelumnya
            if p_url in seen_products:
                continue
            seen_products.add(p_url)
            
            p_price = str(prod.get("price", {}).get("number", 0))
            p_rating = str(prod.get("rating", "0"))
            
            # Cari Lokasi dari toko. 
            shop_city = prod.get("shop", {}).get("city", "")
            p_loc = shop_city if shop_city else "Unknown"
            if p_loc == "Unknown":
                shop_name = prod.get("shop", {}).get("name", "").lower()
                if "tangerang" in shop_name: p_loc = "Tangerang"
                elif "jakarta" in shop_name: p_loc = "Jakarta"
                elif "surabaya" in shop_name: p_loc = "Surabaya"
                elif "bandung" in shop_name: p_loc = "Bandung"
                elif "official" in shop_name: p_loc = "Dilayani Tokopedia"
            
            # Cari "Number Sold" dari labelGroups
            num_sold = "0"
            for label in prod.get("labelGroups", []):
                if label.get("position") == "ri_product_credibility":
                    num_sold_raw = str(label.get("title", ""))
                    num_sold = re.sub(r'[^0-9]', '', num_sold_raw)
                    if not num_sold: num_sold = "0"
            
            print(f"  [{idx+1}/{len(products)}] Ekstrak Ulasan: {p_name[:30]}... ({keyword_reviews_count}/{TARGET_PER_KEYWORD})")
            
            # Anti-Bot Jeda
            time.sleep(random.uniform(1.0, 2.0))
            
            # Ambil ulasan menggunakan URL bersih
            rev_data = fetch_reviews(p_url, limit=REVIEWS_PER_PRODUCT)
            rev_list = rev_data.get("list", [])
            total_review = str(len(rev_list)) if rev_list else "Unknown"
            
            for r in rev_list:
                if keyword_reviews_count >= TARGET_PER_KEYWORD:
                    break # Stop jika sudah mencapai target 1800 untuk keyword ini
                        
                c_review = (r.get("message") or "").strip()
                c_review = c_review.replace("\n", " ").replace("\r", " ")
                    
                if c_review:
                    c_rating = str(r.get("productRating", "0"))
                    all_records.append({
                        "Category": keyword, "Product Name": p_name, "Location": p_loc, "Price": p_price,
                        "Overall Rating": p_rating, "Number Sold": num_sold, "Total Review": total_review,
                        "Customer Rating": c_rating, "Customer Review": c_review
                    })
                    keyword_reviews_count += 1
                        
print("\nSelesai mengekstrak data! Total Semua Baris:", len(all_records))


=== Ekstraksi Keyword: laptop ===
Mode Sort 23: Ditemukan 16 produk.
  [1/16] Ekstrak Ulasan: LENOVO LOQ 15 ARP10E RYZEN 7 7... (0/3333)
  [2/16] Ekstrak Ulasan: MSI THIN 15 I5 12450H RTX2050 ... (18/3333)
  [3/16] Ekstrak Ulasan: LENOVO V14 G4 - AMD RYZEN 5 75... (18/3333)
  [4/16] Ekstrak Ulasan: LENOVO LOQ 15 GeForce RTX 3050... (18/3333)
  [5/16] Ekstrak Ulasan: Lenovo V14 i5-13420H 512GB SSD... (18/3333)
  [6/16] Ekstrak Ulasan: Lenovo Legion LOQ 15IRX9 i7-13... (18/3333)
  [7/16] Ekstrak Ulasan: Acer Aspire 3 A314-42P Ryzen 7... (18/3333)
  [8/16] Ekstrak Ulasan: Lenovo Ideapad Slim 3 Ryzen 7-... (22/3333)
  [9/16] Ekstrak Ulasan: ASUS TUF A15 FA506NFR RYZEN 7 ... (22/3333)
  [10/16] Ekstrak Ulasan: NB LENOVO V14 G5 I5-13420H/8GB... (22/3333)
  [11/16] Ekstrak Ulasan: LENOVO IdeaPad Slim 3 14IRH8 8... (22/3333)
  [12/16] Ekstrak Ulasan: LENOVO V15 G5 IRL i5 13420H 8G... (22/3333)
  [13/16] Ekstrak Ulasan: Lenovo V14 G3 IAP	I3-1215U/8GB... (24/3333)
  [14/16] Ekstrak Ulasan: Lapt

In [13]:
if all_records:
    df = pd.DataFrame(all_records)
    
    # Atur kolom sesuai dataset standar
    columns_order = ['Category', 'Product Name', 'Location', 'Price', 'Overall Rating', 'Number Sold', 'Total Review', 'Customer Rating', 'Customer Review']
    df = df.reindex(columns=columns_order)
    
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\nBerhasil menyimpan {len(df)} baris ulasan ke {OUTPUT_FILE}!")
    display(df.head())
else:
    print("Tidak ada data. Coba perbarui cookie jika kadaluarsa.")


Berhasil menyimpan 38815 baris ulasan ke tokopedia_data_textmining.csv!


,Category,Product Name,Location,Price,Overall Rating,Number Sold,Total Review,Customer Rating,Customer Review
0,laptop,LENOVO LOQ 15 ARP10E RYZEN 7 7735HS RTX3050 6G...,Dilayani Tokopedia,12770010,4.9,100,18,5,"barang luar biasa mantap, seller fast respon, ..."
1,laptop,LENOVO LOQ 15 ARP10E RYZEN 7 7735HS RTX3050 6G...,Dilayani Tokopedia,12770010,4.9,100,18,5,Untuk varian 1 TB ternyata ini 2 drive berbeda...
2,laptop,LENOVO LOQ 15 ARP10E RYZEN 7 7735HS RTX3050 6G...,Dilayani Tokopedia,12770010,4.9,100,18,5,"Pengiriman cepat, pemasangan anti gores rapih,..."
3,laptop,LENOVO LOQ 15 ARP10E RYZEN 7 7735HS RTX3050 6G...,Dilayani Tokopedia,12770010,4.9,100,18,5,sudah dipakai 5 hari pengunnan nyaman dan berf...
4,laptop,LENOVO LOQ 15 ARP10E RYZEN 7 7735HS RTX3050 6G...,Dilayani Tokopedia,12770010,4.9,100,18,5,Respon cepat dan barang sesuai....mantappp
